# qlib regime/window 模型差异分析 v1

目标：验证 `src/analysis` 是否能把模型回测路径转换成 qlib 风格收益分析，并比较不同区间和 regime 下的模型差异。

边界：本 notebook 使用 QuantBench smoke adapter 的回测结果，只验证框架能力；不把 smoke 排名视为论文严格模型优劣。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.analysis import analyze_model_effects_by_regime_and_window

CARD = REPO_ROOT / 'research/analysis_qlib_regime_v1'
MATRIX_CARD = REPO_ROOT / 'research/quantbench_model_matrix_smoke_v1'
CARD.mkdir(parents=True, exist_ok=True)

daily = pd.read_csv(MATRIX_CARD / 'daily_backtest.csv', parse_dates=['date'])
comparison = pd.read_csv(MATRIX_CARD / 'model_comparison.csv')
baseline_model = str(comparison.iloc[0]['model'])
min_date = daily['date'].min()
max_date = daily['date'].max()
mid_date = min_date + (max_date - min_date) / 2
windows = {
    'early_test': (str(min_date.date()), str(mid_date.date())),
    'late_test': (str((mid_date + pd.Timedelta(days=1)).date()), str(max_date.date())),
    'full_test': (str(min_date.date()), str(max_date.date())),
}
baseline_model, windows

('logistic_clf',
 {'early_test': ('2026-05-01', '2026-05-20'),
  'late_test': ('2026-05-21', '2026-06-08'),
  'full_test': ('2026-05-01', '2026-06-08')})

In [2]:
analysis = analyze_model_effects_by_regime_and_window(
    daily,
    windows=windows,
    baseline_model=baseline_model,
)

analysis.qlib_return_frame.to_csv(CARD / 'qlib_return_frame.csv', index=False)
analysis.qlib_risk_metrics.to_csv(CARD / 'qlib_risk_metrics.csv', index=False)
analysis.segment_metrics.to_csv(CARD / 'segment_metrics.csv', index=False)
analysis.model_effect_deltas.to_csv(CARD / 'model_effect_deltas.csv', index=False)
(CARD / 'analysis_summary.json').write_text(
    json.dumps(analysis.summary, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

analysis.summary

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'analysis.regime_window_effect.v1',
 'baseline_model': 'logistic_clf',
 'model_count': 32,
 'regime_count': 4,
 'window_count': 3,
 'qlib_metric_source': 'qlib.contrib.evaluate.risk_analysis',
 'weakest_segments': [{'model': 'mlp_mse',
   'segment_type': 'all',
   'segment': 'all',
   'relative_return_pct_delta': -42.55538514712434,
   'stability_score_delta': -77.95387640388924,
   'diagnosis': '弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。'},
  {'model': 'mlp_mse',
   'segment_type': 'window',
   'segment': 'full_test',
   'relative_return_pct_delta': -42.55538514712434,
   'stability_score_delta': -77.95387640388924,
   'diagnosis': '弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。'},
  {'model': 'mlp_mse',
   'segment_type': 'regime',
   'segment': 'downtrend',
   'relative_return_pct_delta': -28.26075207725995,
   'stability_score_delta': -69.4623142286014,
   'diagnosis': '弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。'},
  {'model': 'alstm_mse',
   'segment_type': 'regime',
 

In [3]:
analysis.qlib_risk_metrics.head(12)

,model,segment_type,segment,return_kind,row_count,metric_source,mean,std,annualized_return,information_ratio,max_drawdown
0,alstm_mse,all,all,excess_return,26,qlib.contrib.evaluate.risk_analysis,-0.011283,0.069567,-9.328355e-01,-2.502130,-0.429431
1,alstm_mse,all,all,strategy_return,26,qlib.contrib.evaluate.risk_analysis,0.014299,0.063322,2.834187e+01,3.483623,-0.187435
2,alstm_mse,regime,bull,excess_return,8,qlib.contrib.evaluate.risk_analysis,-0.025824,0.036067,-9.980245e-01,-11.046051,-0.166917
3,alstm_mse,regime,bull,strategy_return,8,qlib.contrib.evaluate.risk_analysis,0.060076,0.054279,1.072005e+06,17.074961,-0.026762
4,alstm_mse,regime,downtrend,excess_return,10,qlib.contrib.evaluate.risk_analysis,0.005964,0.103592,3.117374e+00,0.888179,-0.158719
5,alstm_mse,regime,downtrend,strategy_return,10,qlib.contrib.evaluate.risk_analysis,0.004192,0.061482,1.706424e+00,1.051888,-0.166972
6,alstm_mse,regime,neutral,excess_return,1,qlib.contrib.evaluate.risk_analysis,0.000000,0.000000,0.000000e+00,0.000000,0.000000
7,alstm_mse,regime,neutral,strategy_return,1,qlib.contrib.evaluate.risk_analysis,0.000000,0.000000,0.000000e+00,0.000000,0.000000
8,alstm_mse,regime,stress,excess_return,7,qlib.contrib.evaluate.risk_analysis,-0.020526,0.040431,-9.928174e-01,-7.832150,-0.159444
9,alstm_mse,regime,stress,strategy_return,7,qlib.contrib.evaluate.risk_analysis,-0.019720,0.058679,-9.912641e-01,-5.184618,-0.057567


In [4]:
analysis.model_effect_deltas.loc[
    analysis.model_effect_deltas['model'] != baseline_model,
    ['segment_type', 'segment', 'model', 'relative_return_pct_delta', 'stability_score_delta', 'diagnosis'],
].sort_values(['relative_return_pct_delta', 'stability_score_delta']).head(20)

,segment_type,segment,model,relative_return_pct_delta,stability_score_delta,diagnosis
31,all,all,mlp_mse,-42.555385,-77.953876,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
223,window,full_test,mlp_mse,-42.555385,-77.953876,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
95,regime,downtrend,mlp_mse,-28.260752,-69.462314,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
136,regime,stress,alstm_mse,-27.940035,-121.123791,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
137,regime,stress,autoformer_mse,-27.940035,-121.123791,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
138,regime,stress,darnn_mse,-27.940035,-121.123791,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
139,regime,stress,dlinear_mse,-27.940035,-121.123791,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
140,regime,stress,fedformer_mse,-27.940035,-121.123791,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
141,regime,stress,gat_mse,-27.940035,-121.123791,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。
142,regime,stress,gcn_mse,-27.940035,-121.123791,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。


## 中文结论

- 做了什么：新增 `src/analysis` qlib 扩展接口，并在本 notebook 中读取 `quantbench_model_matrix_smoke_v1` 的 daily backtest 做 all/regime/window 分层分析。
- 基准模型：`logistic_clf`，窗口：`early_test`、`late_test`、`full_test`。
- 核心输出：`qlib_return_frame.csv`、`qlib_risk_metrics.csv`、`segment_metrics.csv`、`model_effect_deltas.csv`、`analysis_summary.json`。
- 当前最弱分段候选：`mlp_mse / all:all`；这是 smoke adapter 的差异诊断，用来验证框架，不直接作为生产晋升结论。
- 架构结论：具体 qlib/regime/window 计算留在 `src/analysis`；skill 只用于提出 regime 设计、模型差异解释和下一轮假设，不承载项目功能或状态。
- 下一步：把 dashboard 改为读取统一 manifest 后展示这些分析层，并逐步替换 smoke adapter 为更严格的模型实现。